# Lab 3 — PCA and K-means: Ames Housing
Required tasks only (no bonus). Run cells in order. Place this notebook in `L3/` beside your `data/` folder, or edit `DATA_PATH` below.

We group houses without using price as an input. PCA summarizes variation; K-means groups similar houses in the resulting space. This is exploratory clustering, so we fit on the cleaned dataset and use invented houses for inference. Numerical results and plots are generated when run; no Ames results have been invented.

Dependencies: `numpy pandas matplotlib seaborn scikit-learn`. If needed, install them in your notebook environment using `%pip install numpy pandas matplotlib seaborn scikit-learn`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, Normalizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

SEED = 42
DATA_PATH = Path('../data/AmesHousing.csv')
THRESHOLDS = [0.98, 0.95, 0.90, 0.80]
K_VALUES = list(range(2, 11))
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Cleaning and encoding
Reuse the missing-value meaning and outlier rule from Labs 1–2. Keep original features for readable cluster profiles. We do not add the engineered features from Lab2: repeated versions of size/quality would change their weight in distance calculations.

Nominal categories use full one-hot encoding; quality ratings use ordered numerical codes. `MS SubClass` is categorical even though its values are numbers. Missing numerical values use training medians. Price and identifiers never enter PCA or K-means.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place AmesHousing.csv at {DATA_PATH.resolve()} or edit DATA_PATH.')
raw = pd.read_csv(DATA_PATH, sep=None, engine='python')
raw.columns = raw.columns.str.strip()
required = {'SalePrice', 'Gr Liv Area', 'Overall Qual', 'Year Built'}
assert required.issubset(raw.columns), 'Expected original Ames column names with spaces.'
outliers = (raw['Gr Liv Area'] > 4000) & (raw['SalePrice'] < 300000)
df = raw.loc[~outliers].reset_index(drop=True).copy()
X_raw = df.drop(columns=['SalePrice', 'Order', 'PID'], errors='ignore')
print(f'{len(df)} houses; removed {outliers.sum()} outliers; {X_raw.shape[1]} raw features.')

ABSENCE = ['Alley', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1',
           'BsmtFin Type 2', 'Fireplace Qu', 'Garage Type', 'Garage Finish',
           'Garage Qual', 'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature', 'Mas Vnr Type']
QUALITY = ['Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond', 'Heating QC',
           'Kitchen Qual', 'Fireplace Qu', 'Garage Qual', 'Garage Cond', 'Pool QC']
def clean_features(frame):
    x = frame.copy()
    for col in ABSENCE:
        if col in x: x[col] = x[col].fillna('None')
    for col in QUALITY:
        if col in x:
            x[col] = x[col].map({'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5})
    if 'MS SubClass' in x: x['MS SubClass'] = x['MS SubClass'].astype(str)
    return x

X = clean_features(X_raw)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in X if col not in num_cols]
# Log only nonnegative, strongly right-skewed numerical measurements; exclude ratings and years.
log_candidates = ['Lot Frontage', 'Lot Area', 'Mas Vnr Area', 'BsmtFin SF 1',
                  'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF',
                  '2nd Flr SF', 'Gr Liv Area', 'Garage Area', 'Wood Deck SF',
                  'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch',
                  'Pool Area', 'Misc Val']
LOG_COLS = [col for col in log_candidates if col in X and X[col].min() >= 0 and X[col].skew() > 1]
print('Log1p features:', LOG_COLS)

def make_preprocessor():
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), num_cols),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ], verbose_feature_names_out=False)

def before_encoding(frame, use_log):
    x = clean_features(frame)
    if use_log: x[LOG_COLS] = np.log1p(x[LOG_COLS])
    return x


## 2. PCA and preprocessing experiments
Compare a 2×2 design: min-max scaling or row-wise L2 normalization, each with and without `log1p`. Here normalization explicitly means dividing each encoded row by its Euclidean length; it is not standardization. Large measurements can dominate that variant, which is part of the comparison.

For each variant, fit PCA once and keep the smallest number of components reaching each threshold. The cumulative plot reports retained variance, not prediction accuracy.

In [ ]:
experiments = {}
variance_rows = []
for use_log in [False, True]:
    for scaling in ['minmax', 'normalize']:
        name = scaling + (' + log1p' if use_log else '')
        prep = make_preprocessor()
        encoded = prep.fit_transform(before_encoding(X_raw, use_log))
        scaler = MinMaxScaler() if scaling == 'minmax' else Normalizer(norm='l2')
        scaled = scaler.fit_transform(encoded)
        pca = PCA(svd_solver='full')
        scores = pca.fit_transform(scaled)
        cumulative = np.cumsum(pca.explained_variance_ratio_)
        counts = {t: int(np.searchsorted(cumulative, t) + 1) for t in THRESHOLDS}
        experiments[name] = dict(prep=prep, scaler=scaler, pca=pca, scores=scores,
                                 counts=counts, use_log=use_log)
        for t, count in counts.items():
            variance_rows.append([name, t, encoded.shape[1], count, cumulative[count-1]])
        display(pd.DataFrame({'PC': np.arange(1, len(cumulative)+1),
                              'explained_ratio': pca.explained_variance_ratio_,
                              'cumulative': cumulative}).rename_axis(name))
        plt.plot(np.arange(1, len(cumulative)+1), cumulative, label=name)
for t in THRESHOLDS: plt.axhline(t, color='gray', linestyle=':', alpha=.5)
plt.xlabel('Number of principal components'); plt.ylabel('Cumulative explained variance')
plt.legend(); plt.tight_layout(); plt.show()
variance_table = pd.DataFrame(variance_rows, columns=['variant', 'threshold', 'encoded_features', 'components', 'retained_variance'])
display(variance_table)


## 3. Choose cluster counts: elbow and silhouette
Fit candidate values of k for every variant and variance threshold. Elbow plots include k=1 as a reference. Silhouette scores use the full retained PCA space. Higher scores indicate better separation, but do not prove that the groups have useful meanings.

The default choice maximizes silhouette score over the tested range. Inspect the elbow and silhouette plots before accepting it. Set `K_OVERRIDES[(variant, threshold)] = k` below and rerun this section to make a justified manual choice. Inertia values from different preprocessing spaces should not be compared directly.

In [ ]:
K_OVERRIDES = {}  # Example: {('minmax', 0.98): 3}
results = {}; comparison_rows = []
for name, exp in experiments.items():
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for col, t in enumerate(THRESHOLDS):
        z = exp['scores'][:, :exp['counts'][t]]
        models = {}; inertias = []; sil_scores = []
        for k in [1] + K_VALUES:
            model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(z)
            models[k] = model; inertias.append(model.inertia_)
            if k > 1: sil_scores.append(silhouette_score(z, model.labels_))
        best_k = K_OVERRIDES.get((name, t), K_VALUES[int(np.argmax(sil_scores))])
        assert best_k in K_VALUES
        results[(name, t)] = dict(z=z, models=models, k=best_k, model=models[best_k])
        comparison_rows.append([name, t, z.shape[1], best_k, sil_scores[K_VALUES.index(best_k)]])
        axes[0,col].plot([1]+K_VALUES, inertias, 'o-')
        axes[0,col].set(title=f'{t:.0%} variance', xlabel='k', ylabel='Inertia')
        axes[1,col].plot(K_VALUES, sil_scores, 'o-')
        axes[1,col].axvline(best_k, color='red', linestyle='--')
        axes[1,col].set(xlabel='k', ylabel='Mean silhouette')
    fig.suptitle(name); plt.tight_layout(); plt.show()
comparison = pd.DataFrame(comparison_rows, columns=['variant', 'threshold', 'components', 'k', 'silhouette'])
display(comparison)


In [ ]:
# Full silhouette diagrams for several k values in every representation.
for (name, t), result in results.items():
    candidates = sorted(set([2, 3, 4, result['k']]))
    fig, axes = plt.subplots(1, len(candidates), figsize=(4*len(candidates), 4), squeeze=False)
    for ax, k in zip(axes[0], candidates):
        labels = result['models'][k].labels_
        values = silhouette_samples(result['z'], labels)
        bottom = 0
        for group in range(k):
            v = np.sort(values[labels == group]); top = bottom + len(v)
            ax.fill_betweenx(np.arange(bottom, top), 0, v, alpha=.7)
            ax.text(-.08, (bottom+top)/2, str(group))
            bottom = top + 10
        ax.axvline(values.mean(), color='red', linestyle='--')
        ax.set(title=f'k={k}, mean={values.mean():.3f}', xlabel='Silhouette', ylabel='Houses by cluster', xlim=(-1,1))
    fig.suptitle(f'{name}, retained variance {t:.0%}'); plt.tight_layout(); plt.show()


## 4. Two-dimensional cluster views
Fit in all retained dimensions, display PC1 and PC2. Centers are projected onto these same axes. If a threshold retains only one component, use zero on the second axis. Overlap in 2D does not necessarily mean overlap in the full clustering space.

In [ ]:
for name, exp in experiments.items():
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    for ax, t in zip(axes, THRESHOLDS):
        r = results[(name, t)]; z = r['z']; model = r['model']
        y = z[:,1] if z.shape[1] > 1 else np.zeros(len(z))
        centers = model.cluster_centers_
        cy = centers[:,1] if z.shape[1] > 1 else np.zeros(len(centers))
        ax.scatter(z[:,0], y, c=model.labels_, cmap='tab10', s=10, alpha=.5)
        ax.scatter(centers[:,0], cy, c='black', marker='X', s=140, label='Centers')
        ax.set(title=f'{t:.0%}: {z.shape[1]} PCs, k={r["k"]}', xlabel='PC1', ylabel='PC2')
        ax.legend()
    fig.suptitle(name); plt.tight_layout(); plt.show()
# ARI compares memberships without requiring matching cluster numbers.
reference = results[('minmax', .98)]['model'].labels_
comparison['ARI_vs_minmax_98'] = [adjusted_rand_score(reference, results[(row.variant,row.threshold)]['model'].labels_)
                                 for row in comparison.itertuples()]
display(comparison)


**Interpretation:** Compare the chosen k, silhouette plots, component counts, and membership agreement (ARI). Logarithms compress very large measurements. Min-max rescales columns independently; normalization rescales entire houses and can suppress absolute size differences. PCA preserves variance, which does not guarantee preservation of cluster separation. Cluster numbers are arbitrary across runs. Use the actual tables to describe which variants changed most.

## 5. Which features influence the leading components?
Use the formula requested in the assignment. Large absolute values show stronger contributions; the sign shows direction within a component. PCA signs can flip without changing its meaning. These loadings describe the transformed feature space and are not automatically correlations with unscaled original variables.

In [ ]:
for name, exp in experiments.items():
    pca = exp['pca']; names = exp['prep'].get_feature_names_out()
    count = min(5, len(pca.components_))
    loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), index=names)
    top = loadings.iloc[:,:count].abs().max(axis=1).nlargest(18).index
    plt.figure(figsize=(9,8))
    sns.heatmap(loadings.loc[top].iloc[:,:count], cmap='coolwarm', center=0,
                xticklabels=[f'PC{i+1}' for i in range(count)])
    plt.title(f'Leading PCA loadings — {name}'); plt.tight_layout(); plt.show()
    for pc in range(min(3, count)):
        print(name, f'PC{pc+1}:', ', '.join(loadings.iloc[:,pc].abs().nlargest(5).index))


## 6. Describe the clusters using original house characteristics
Summarize all selected clusterings. The detailed plots use min-max with 98% variance as a reference; change `PROFILE_KEY` to inspect another result. Price is used only after clustering to describe groups. These are descriptive associations, not causal effects.

In [ ]:
PROFILE_COLS = ['Gr Liv Area', 'Lot Area', 'Overall Qual', 'Year Built', 'SalePrice']
for key, r in results.items():
    profile = df.assign(cluster=r['model'].labels_)
    print(key)
    display(profile.groupby('cluster')[PROFILE_COLS].median().round(1).join(profile.groupby('cluster').size().rename('count')))
    for group, houses in profile.groupby('cluster'):
        med = houses[PROFILE_COLS].median()
        neighborhood = houses['Neighborhood'].mode().iloc[0]
        print(f'Cluster {group}: {len(houses)} houses; median living area {med["Gr Liv Area"]:.0f} sq ft, '
              f'quality {med["Overall Qual"]:.1f}, construction year {med["Year Built"]:.0f}, '
              f'price ${med["SalePrice"]:,.0f}; most common neighborhood {neighborhood}.')
PROFILE_KEY = ('minmax', .98)
profile = df.assign(cluster=results[PROFILE_KEY]['model'].labels_)
fig, axes = plt.subplots(1, 4, figsize=(18,4))
for ax, feature in zip(axes, ['Gr Liv Area', 'Overall Qual', 'Year Built', 'SalePrice']):
    sns.boxplot(data=profile, x='cluster', y=feature, ax=ax)
plt.tight_layout(); plt.show()
plt.figure(figsize=(12,5))
sns.heatmap(pd.crosstab(profile['cluster'], profile['Neighborhood'], normalize='index'), cmap='Blues')
plt.title('Neighborhood proportions within each cluster'); plt.tight_layout(); plt.show()


## 7. Invented houses and inference
Build three new examples by adapting complete real-house templates. This preserves coherent values for the many fields not explicitly invented. The changed areas and qualities describe a modest, medium, and larger house; these are synthetic variants, not independent test observations.

Apply the already-fitted encoder, scaler and PCA, retaining the same component count. Never fit preprocessing on these houses. K-means centers are in PCA space, so raw house measurements cannot be compared directly to them.

In [ ]:
# Start with low-, middle-, and high-size templates, then invent new measurements.
ordered = X_raw.sort_values('Gr Liv Area')
new_houses = ordered.iloc[[int((len(ordered)-1)*q) for q in [.2,.5,.8]]].copy()
new_houses.index = ['Modest house', 'Medium house', 'Large house']
for label, factor, quality in zip(new_houses.index, [.92, 1.03, 1.10], [4,6,8]):
    for col in ['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF']:
        new_houses.loc[label,col] = round(new_houses.loc[label,col] * factor)
    new_houses.loc[label,'Gr Liv Area'] = new_houses.loc[label,['1st Flr SF','2nd Flr SF','Low Qual Fin SF']].sum()
    new_houses.loc[label,'Overall Qual'] = quality
    new_houses.loc[label,'Lot Area'] = round(new_houses.loc[label,'Lot Area'] * factor)
display(new_houses[['Gr Liv Area','Lot Area','Overall Qual','Year Built','Neighborhood']])
assignments = pd.DataFrame(index=new_houses.index)
for (name,t), r in results.items():
    exp = experiments[name]
    encoded = exp['prep'].transform(before_encoding(new_houses, exp['use_log']))
    scaled = exp['scaler'].transform(encoded)
    z_new = exp['pca'].transform(scaled)[:, :exp['counts'][t]]
    assignments[f'{name} / {t:.0%}'] = r['model'].predict(z_new)
display(assignments)
# Inspect assigned reference-cluster medians next to each synthetic house.
labels = assignments[f'{PROFILE_KEY[0]} / {PROFILE_KEY[1]:.0%}']
medians = profile.groupby('cluster')[PROFILE_COLS].median()
for house, group in labels.items():
    print(f'{house} -> cluster {group}')
    display(pd.DataFrame({'invented_house': new_houses.loc[house, PROFILE_COLS[:-1]],
                          'cluster_median': medians.loc[group, PROFILE_COLS[:-1]]}))


## 8. Conclusions to complete after running
Use your generated results to write a few sentences in your own words:

1. How many components were needed for 98%, 95%, 90% and 80% variance?
2. Which k does the elbow suggest? Does silhouette support it? Explain any override or ambiguous elbow.
3. Which preprocessing choice gives the clearest groups? Do logarithms change membership or separation?
4. Which original features dominate PC1 and PC2? How does this explain the scatter plots?
5. Describe each reference cluster using the numerical summaries above, and explain the differences between variants.
6. Do the invented houses fit the characteristics of their assigned clusters? Mention specific measurements, including any surprising assignment.

A high silhouette value alone is not enough to call a model best. Consider cluster sizes, interpretability and preprocessing effects. Raw inertia across different spaces is not a fair quality ranking.